In [198]:
import torch
import torch.nn.functional as F
import requests

In [199]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()

print(words[:10])
len(words)

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


32033

In [214]:
def stoi(s):
    return {ch: i for i, ch in enumerate(".abcdefghijklmnopqrstuvwxyz")}[s]
def getXY(word, block_size):
    X = []
    Y = []
    context = [0] * block_size
    for ch in word + ".":
        X.append(context)
        Y.append(stoi(ch))
        ix = stoi(ch)
        context = context[1:] + [ix]
        print(context)
    return torch.tensor(X), torch.tensor(Y)

X, Y = getXY("hello", 8)



[0, 0, 0, 0, 0, 0, 0, 8]
[0, 0, 0, 0, 0, 0, 8, 5]
[0, 0, 0, 0, 0, 8, 5, 12]
[0, 0, 0, 0, 8, 5, 12, 12]
[0, 0, 0, 8, 5, 12, 12, 15]
[0, 0, 8, 5, 12, 12, 15, 0]


In [215]:
print(X.shape)
print(X)
print(Y.shape)
print(Y)

torch.Size([6, 8])
tensor([[ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  8],
        [ 0,  0,  0,  0,  0,  0,  8,  5],
        [ 0,  0,  0,  0,  0,  8,  5, 12],
        [ 0,  0,  0,  0,  8,  5, 12, 12],
        [ 0,  0,  0,  8,  5, 12, 12, 15]])
torch.Size([6])
tensor([ 8,  5, 12, 12, 15,  0])


In [216]:
C = torch.randn((27, 2))
hot = F.one_hot(Y, num_classes=27).float()
hot @ C

tensor([[-0.1236,  0.7157],
        [ 0.3878,  0.6760],
        [ 0.1701,  0.9408],
        [ 0.1701,  0.9408],
        [-0.5617,  0.0131],
        [ 1.7475,  1.9435]])

In [217]:
emb = C[X]
print(emb.shape)
print(emb)

torch.Size([6, 8, 2])
tensor([[[ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435]],

        [[ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [-0.1236,  0.7157]],

        [[ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [-0.1236,  0.7157],
         [ 0.3878,  0.6760]],

        [[ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [ 1.7475,  1.9435],
         [-0.1236,  0.7157],
         [ 0.3878,  0.6760],
         [ 0.1701,  0.9408]],

        [[ 1.7475,  1.9435],
         [ 1.

In [238]:
W1 = torch.randn((16, 100), requires_grad=True)
b1 = torch.randn(100, requires_grad=True)
h = emb.view(-1, 16) @ W1 + b1
print(h.shape)


torch.Size([6, 100])


In [239]:
W2 = torch.randn((100, 27), requires_grad=True)
b2 = torch.randn(27, requires_grad=True)
logits = h @ W2 + b2
print(logits.shape)

torch.Size([6, 27])


In [ ]:
#overfit test
h = emb.view(-1, 16) @ W1 + b1
logits = h @ W2 + b2
#backwords
loss = F.cross_entropy(logits, Y)
loss.backward()
#forward
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)
optimizer.step()
print(loss)
optimizer.zero_grad()


tensor(0.8571, grad_fn=<NllLossBackward0>)
